In [1]:
import sys
import os
sys.path.append(os.path.abspath("") + "/../ros2_ws/src/icm20948")

In [2]:
from icm20948.icm_wrapper import imuDataGet, imuInit
imuInit()

-1787887312

In [3]:

imuDataGet()

icm20948AccelRead took 0.001909 seconds
icm20948GyroRead took 0.002541 seconds
icm20948MagRead took 0.008496 seconds
imuAHRSupdate took 0.000002 seconds
Total imuDataGet took 0.013022 seconds


(<icm20948.icm_wrapper.IMU_ST_ANGLES_DATA at 0xffff9402add0>,
 <icm20948.icm_wrapper.IMU_ST_SENSOR_DATA at 0xffff9402b050>)

In [4]:
%%timeit

imuDataGet()

icm20948AccelRead took 0.002438 seconds
icm20948GyroRead took 0.001611 seconds
icm20948MagRead took 0.009186 seconds
imuAHRSupdate took 0.000002 seconds
Total imuDataGet took 0.013293 seconds
icm20948AccelRead took 0.001667 seconds
icm20948GyroRead took 0.001535 seconds
icm20948MagRead took 0.008420 seconds
imuAHRSupdate took 0.000001 seconds
Total imuDataGet took 0.011675 seconds
icm20948AccelRead took 0.001536 seconds
icm20948GyroRead took 0.002339 seconds
icm20948MagRead took 0.008669 seconds
imuAHRSupdate took 0.000001 seconds
Total imuDataGet took 0.012560 seconds
icm20948AccelRead took 0.001547 seconds
icm20948GyroRead took 0.002653 seconds
icm20948MagRead took 0.008401 seconds
imuAHRSupdate took 0.000001 seconds
Total imuDataGet took 0.012629 seconds
icm20948AccelRead took 0.001544 seconds
icm20948GyroRead took 0.001547 seconds
icm20948MagRead took 0.008417 seconds
imuAHRSupdate took 0.000000 seconds
Total imuDataGet took 0.011524 seconds
icm20948AccelRead took 0.001548 seconds


In [2]:
# Run 1000 cycles to let it calibrate
for _ in range(1000):
    imuDataGet()

KeyboardInterrupt: 

In [7]:
import numpy as np
import time

DT = 0.01  # 10ms update rate

class IMU_Estimator:
    def __init__(self):
        self.position = np.zeros(3)  # X, Y, Z
        self.velocity = np.zeros(3)  # Vx, Vy, Vz
        self.orientation = np.zeros(3)  # Roll, Pitch, Yaw

    def update(self, angles, accel):
        """ Process IMU data and estimate orientation + position """

        # ✅ 3️⃣ Integrate gyro to estimate orientation (Euler method)
        self.orientation = np.array([angles.roll, angles.pitch, angles.yaw])

        # ✅ 6️⃣ Convert accel to m/s² (1g = 9.81 m/s²)
        accel *= 9.81

        # ✅ 7️⃣ Integrate acceleration to estimate velocity & position
        self.velocity += accel * DT
        self.position += self.velocity * DT

        # 🔹 Print results
        print(f"Position: {self.position.round(2)}, Orientation: {self.orientation.round(2)}")

# ✅ Initialize estimator
imu_estimator = IMU_Estimator()

# ✅ Run simulation with IMU data
for _ in range(1000):  # Simulate 10 seconds (1000 cycles @ 10ms)
    angles, q, gyro, accel, grav, magn = imuDataGet()  # Read IMU data
    accel = np.array(accel.to_array()) - np.array(grav.to_array())
    imu_estimator.update(angles, accel)
    time.sleep(DT)  # Simulate real-time updates

Position: [-0. -0. -0.], Orientation: [-46.07  -2.02 -87.1 ]
Position: [-0. -0. -0.], Orientation: [-46.06  -2.14 -87.04]
Position: [-0. -0. -0.], Orientation: [-46.09  -2.26 -86.95]
Position: [-0. -0. -0.], Orientation: [-46.12  -2.41 -86.83]
Position: [-0. -0. -0.], Orientation: [-46.11  -2.26 -86.93]
Position: [-0. -0. -0.], Orientation: [-46.18  -2.37 -86.77]
Position: [-0. -0. -0.], Orientation: [-46.3   -2.13 -86.77]
Position: [-0. -0. -0.], Orientation: [-46.37  -2.21 -86.61]
Position: [-0. -0. -0.], Orientation: [-46.42  -2.05 -86.67]
Position: [-0. -0. -0.], Orientation: [-46.47  -2.13 -86.52]
Position: [-0. -0. -0.], Orientation: [-46.5   -2.01 -86.61]
Position: [-0. -0. -0.], Orientation: [-46.48  -1.96 -86.72]
Position: [-0. -0. -0.], Orientation: [-46.49  -2.05 -86.61]
Position: [-0. -0. -0.], Orientation: [-46.45  -2.   -86.73]
Position: [-0. -0. -0.], Orientation: [-46.35  -2.   -86.87]
Position: [-0. -0. -0.], Orientation: [-46.29  -1.97 -86.95]
Position: [-0. -0. -0.],

KeyboardInterrupt: 

[0.02789306640625, -1.019775390625, 0.0467529296875]